# **IBM HR Analytics Employee Attrition & Performance**💼📉

**Goal:** Predict Employee Attrition taking into account Data Privacy

**Dataset:** https://www.kaggle.com/datasets/pavansubhasht/ibm-hr-analytics-attrition-dataset

# Installs and Imports

In [ ]:
# Instal SDV, Optuna
!pip install -q sdv
!pip install -q optuna
!pip install -q imbalanced-learn

# Standard Library Imports
import os
import itertools
import warnings

# Third-Party Library Imports
try:
    import numpy as np
    import pandas as pd
    import seaborn as sns
    import matplotlib.pyplot as plt
    import optuna
    from optuna import trial
    from sklearn.model_selection import StratifiedKFold, train_test_split
    from sklearn.preprocessing import LabelEncoder
    from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, roc_auc_score
    import lightgbm as lgb
    from lightgbm import LGBMClassifier, plot_importance
    from imblearn.over_sampling import SMOTE

    # SDV Library Imports
    from sdv.metadata import SingleTableMetadata
    from sdv.single_table import GaussianCopulaSynthesizer
    from sdv.evaluation.single_table import evaluate_quality, run_diagnostic, get_column_plot, get_column_pair_plot
    from sdv.sampling import Condition

    # Ignore warnings
    warnings.filterwarnings("ignore")

    # Set display options for Pandas
    pd.set_option("display.float_format", "{:.2f}".format)
    pd.set_option("display.max_columns", 80)
    pd.set_option("display.max_rows", 80)

    # Print a confirmation message
    print("Modules loaded")
except ImportError as e:
    print(f"Error: {e}")
    print("Failed to load all modules.")

# EDA

In [ ]:
df = pd.read_csv('employee_attrition.csv')

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

## Drop based on Unique

In [ ]:
col_unique = df.nunique().sort_values()
col_unique

We observe that the columns 'EmployeeCount,' 'Over18,' and 'StandardHours' have only one unique value each, while 'EmployeeNumber' has 1470 unique values. Since these features do not provide meaningful variation, we have decided to remove these columns from the dataset.

In [ ]:
df.drop(['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours'], axis="columns", inplace=True)

<b>Selecting the threshold for determining what should be considered as categorical</b>

In this context, we are addressing the decision-making process involved in defining the criteria or threshold that distinguishes whether a variable should be treated as categorical. The threshold serves as a guideline for categorizing variables based on certain characteristics or properties, such as the number of unique values or the nature of the data. This decision is crucial in data analysis and preprocessing as it influences subsequent encoding and treatment of variables in statistical or machine learning models.

In [ ]:
cat_treshold = 20

## Explore Based on Types

In [ ]:
catvars_complex = []
catvars = []
numvars = []
ordvars = []
target= 'Attrition'
for column in df.columns:
    if column == target:
        continue
    if df[column].dtype == object:
        if col_unique[column] > cat_treshold:
            catvars_complex.append(column)
        else:
            catvars.append(column)
    else:
        if col_unique[column] > cat_treshold:
            numvars.append(column)
        else:
            ordvars.append(column)

In [ ]:
print('Catvars Complex:', catvars_complex)
print('\nCatvars:', catvars)
print('\nNumerical:', numvars)
print('\nOrdvars:', ordvars)

## Numericals

In [ ]:
# Analyzing Numerical Variables
sns.set(style="whitegrid")

fig, axes = plt.subplots(nrows=len(numvars), ncols=1, figsize=(10, len(numvars)*5))
for i, var in enumerate(numvars):
    sns.histplot(df[var], ax=axes[i])
    axes[i].set_title(f'Distribution of {var}')
plt.tight_layout()
plt.show()

## Correlation Matrix

In [ ]:
sns.heatmap(df[numvars].corr(), annot=True, fmt=".2f")
plt.show()

## Categoricals and Ordinal Variables

The use of Pandas .to_dict() is encountering issues when applied in Seaborn for countplots, barplots, and histograms with count. To address this, we created a dictionary directly from the DataFrame and directly passed it to these Seaborn functions.


In [ ]:
cat_ord_vars = catvars + ordvars
cat_ord_size = len(cat_ord_vars)
cat_ord_dict = {}
for key in cat_ord_vars:
    cat_ord_dict[key] = df[key].values

In [ ]:
fig, axes = plt.subplots(nrows=cat_ord_size, ncols=1, figsize=(10, cat_ord_size*4))
for i, var in enumerate(cat_ord_vars):
    sns.countplot(x=var, data=cat_ord_dict, ax=axes[i])
    axes[i].set_title(f'Distribution of {var}', fontsize=14, fontweight='bold')  # Adjust title size and boldness
    axes[i].tick_params(axis='x', rotation=45, labelsize=10)  # Rotate x-axis labels to avoid label overlap

plt.tight_layout()
plt.show()


### Grouping Categorical and Ordinals  


It might be essential to group categorical or ordinal variables based on the distribution of their unique values to prevent potential issues related to quasi-identifiers. This involves assessing the distribution of unique values within these variables and, if necessary, combining or grouping certain categories or levels. This process aims to enhance the quality of the data and improve the interpretability of the variables, especially when dealing with a large number of unique values or uneven distributions within categorical or ordinal features.

### Encoding to Numeric



*Label Encoding and Ordinal Encoding for Categorical Variables with Few Unique Values:*

For categorical variables with a limited number of unique values, such as those representing complexity, we can use techniques like Label Encoding or Ordinal Encoding.

**complex -> LabelEncoder() , OrdinalEncoder()**

**few dummies -> pd.get_dummies(df, columns=dummy_col, drop_first=True, dtype='uint8')**

In [ ]:
le = LabelEncoder()
for col in catvars:
    df[col] = le.fit_transform(df[col])

## Target

In [ ]:
# Create a bar plot for the 'Attrition' feature
plt.figure(figsize=(8, 6))
ax = sns.countplot(x='Attrition', data=df, palette='viridis')

# Add total counts and percentages on top of the bars
total_counts = len(df)
for p in ax.patches:
    height = p.get_height()
    ax.annotate(f'{int(height)}\n{height/total_counts*100:.2f}%',
                (p.get_x() + p.get_width() / 2., height),
                ha='center', va='center', xytext=(0, 10),
                textcoords='offset points', fontsize=12, color='black')

plt.title('Attrition Count')
plt.xlabel('Attrition')
plt.ylabel('Count')
plt.show()

Observing a noticeable imbalance in the target variable, it becomes imperative to address this during the data splitting process, emphasizing the importance of a Stratified Split approach.

To mitigate the challenge posed by the imbalance, various techniques can be employed.

In this instance, we utilized <b>Stratified K-Fold</b> for splitting and subsequently employed <b>Synthetic Data Generation with SDV.</b>


### Against Target

In [ ]:
# same to_dict problem
df_cols = df.columns
df_size = len(df_cols)
df_dict = {}
for key in df_cols:
    df_dict[key] = df[key].values

In [ ]:
cat_ord_target_dict = {**cat_ord_dict, target: df[target].values}

In [ ]:
fig, axes = plt.subplots(nrows=cat_ord_size, ncols=1, figsize=(10, cat_ord_size*4))
for i, var in enumerate(cat_ord_dict):
    sns.histplot(
        data=cat_ord_target_dict, x=var,  hue=target,
        binwidth=0.5, stat="count", multiple="stack",
        ax=axes[i]
    )
plt.tight_layout()
plt.show()

## Evaluate Formula

In [ ]:
def evaluate(model, X_train, X_test, y_train, y_test):
    label_encoder = LabelEncoder()

    # Fit and transform the label encoder on the combined set of labels
    all_labels = pd.concat([y_train, y_test])
    label_encoder.fit(all_labels)

    # Convert labels to numerical values
    y_train_encoded = label_encoder.transform(y_train)
    y_test_encoded = label_encoder.transform(y_test)

    # Predictions on training and testing data
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Convert predicted labels to numerical values
    y_train_pred_numeric = label_encoder.transform(y_train_pred)
    y_test_pred_numeric = label_encoder.transform(y_test_pred)

    print("TRAINING RESULTS: \n===============================")
    clf_report_train = pd.DataFrame(classification_report(y_train_encoded, y_train_pred_numeric, output_dict=True))
    print(f"CONFUSION MATRIX:\n{confusion_matrix(y_train_encoded, y_train_pred_numeric)}")
    print(f"\nACCURACY SCORE:\n{accuracy_score(y_train_encoded, y_train_pred_numeric):.4f}")
    print(f"\nCLASSIFICATION REPORT:\n{clf_report_train}")

    print("\n\n\nTESTING RESULTS: \n===============================")
    clf_report_test = pd.DataFrame(classification_report(y_test_encoded, y_test_pred_numeric, output_dict=True))
    print(f"CONFUSION MATRIX:\n{confusion_matrix(y_test_encoded, y_test_pred_numeric)}")
    print(f"\nACCURACY SCORE:\n{accuracy_score(y_test_encoded, y_test_pred_numeric):.4f}")
    print(f"\nCLASSIFICATION REPORT:\n{clf_report_test}")

## Training

In [ ]:
def train_test(data: pd.DataFrame, categoricals: list[str, ], target: str):
    X = data.drop(target, axis=1)
    y = data[target]
    X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    dtrain = lgb.Dataset(X_train, label=y_train, categorical_feature=categoricals)
    dvalid = lgb.Dataset(X_valid, label=y_valid, categorical_feature=categoricals)
    return (dtrain, dvalid)

## Tuning LightGBM with Stratified Split

with Stratified Kfold (due to imbalance)

In [ ]:
def skf(data: pd.DataFrame, categoricals: list[str, ], target: str):
    datasets = []
    X = data.drop(target, axis=1)
    y = data[target]
    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    for train_index, valid_index in kf.split(X, y):
        train_data, valid_data = X.iloc[train_index], X.iloc[valid_index]
        train_target, valid_target = y.iloc[train_index], y.iloc[valid_index]

        dtrain = lgb.Dataset(train_data, label=train_target, categorical_feature=categoricals)
        dvalid = lgb.Dataset(valid_data, label=valid_target, categorical_feature=categoricals)

        datasets.append((dtrain, dvalid))
    return datasets

def lgb_objective(trial, datasets):
    scores = []
    for dtrain, dvalid in datasets:
        model = LGBMClassifier(
            learning_rate=trial.suggest_float('learning_rate', 0.001, 0.3, log=True),
            n_estimators=trial.suggest_int('n_estimators', 50, 1200, 25),
            max_depth=trial.suggest_int('max_depth', 1, 15),
            num_leaves=trial.suggest_int('num_leaves', 10, 150),
            colsample_bytree=trial.suggest_float('colsample_bytree', 0.3, 1.0),
            feature_fraction=trial.suggest_float('feature_fraction', 0.3, 1.0),
            reg_lambda=trial.suggest_float('reg_lambda', 0.0, 1.0),
            reg_alpha=trial.suggest_float('reg_alpha', 0.0, 1.0),
            min_split_gain=trial.suggest_float('min_split_gain', 0.0001, 0.1),
            verbose=-1
        )
        early_stopping_callback = lgb.early_stopping(stopping_rounds=20, verbose=True)
        model.fit(
            dtrain.data, dtrain.label,
            eval_set=[(dtrain.data, dtrain.label), (dvalid.data, dvalid.label)],
            eval_metric='auc',
             callbacks=[early_stopping_callback]
        )

        preds = model.predict_proba(dvalid.data)[:, 1]
        score = roc_auc_score(dvalid.label, preds)  # Use test_y instead of y_val
        scores.append(score)
        print('score', score)
    return np.mean(scores)


## Modelling Real Data

In [ ]:
datasets = skf(df.copy(), catvars, target)

study = optuna.create_study(direction='maximize')
study.optimize(lambda trial: lgb_objective(trial, datasets.copy()), n_trials=5, n_jobs=2)
best_params = study.best_params
best_trial = study.best_trial

In [ ]:
print(f"Best trial - Loss: {best_trial.value:.4f}, \n Params: {best_params}")

In [ ]:
fig = optuna.visualization.plot_param_importances(study)
fig.show()

In [ ]:
best_params

In [ ]:
train, test = train_test(df.copy(), catvars, target)
model = LGBMClassifier(**best_params, verbose=-1)
early_stopping_callback = lgb.early_stopping(stopping_rounds=20, verbose=True)
model.fit(train.data, train.label, eval_metric='auc')

In [ ]:
# Combine features and target into new DataFrame
df_train = pd.concat([train.data, train.label], axis=1)
df_test = pd.concat([test.data, test.label], axis=1)

df_train.to_csv('df_train.csv', index=False)
df_test.to_csv('df_test.csv', index=False)

In [ ]:
# Save test.data and test.label to provide comparative analysis with the results from anonymization and synthetic data
test_data_compare = test.data
test_label_compare = test.label

In [ ]:
evaluate(model, train.data, test.data, train.label, test.label)

In [ ]:
plot_importance(model, importance_type='split', figsize=(10, 8))
plt.show()

# Synthetic Data

## Generation of Synthetic Data Using SDV

In [ ]:
df.head()

In [ ]:
metadata = SingleTableMetadata()

In [ ]:
metadata.detect_from_dataframe(df)

In [ ]:
metadata.visualize()

In [ ]:
synthesizer = GaussianCopulaSynthesizer(metadata)

In [ ]:
synthesizer.fit(
    data=df
)

In [ ]:
synthetic_data = synthesizer.sample(
    num_rows=4000
)

synthetic_data.head()

In [ ]:
diagnostic = run_diagnostic(
    real_data=df,
    synthetic_data=synthetic_data,
    metadata=metadata
)

In [ ]:
quality_report = evaluate_quality(
    df,
    synthetic_data,
    metadata
)

In [ ]:
quality_report.get_details('Column Shapes')

In [ ]:
for col in df.columns:
    fig = get_column_plot(
        real_data=df,
        synthetic_data=synthetic_data,
        column_name= col,
        metadata=metadata
    )

    fig.show()

In [ ]:
comparison_column = 'Attrition'

for col in df.columns:
    if col != comparison_column:
        fig = get_column_pair_plot(
            real_data=df,
            synthetic_data=synthetic_data,
            column_names=[comparison_column, col],
            metadata=metadata
        )

        fig.show()

**First Report:**
* Data Validity: 100.0%

This indicates that the synthetic data generated by SDV is valid, meaning it meets the constraints and validity checks specified for the original dataset. All generated records are within the acceptable range and adhere to the rules defined for the data.

* Data Structure: 100.0%

This score suggests that the synthetic data preserves the overall structure of the original data. The relationships, dependencies, and patterns in the data are accurately captured by the synthetic data.




**Second Report:**
* Column Shapes: 93.11%

This score evaluates how well the synthetic data reproduces the shapes (distributions) of individual columns. A high score indicates that the statistical properties of each column in the synthetic data are close to those in the original data.

* Column Pair Trends: 87.89%

This score assesses how well the relationships between pairs of columns are preserved in the synthetic data. It looks at trends and correlations between pairs of columns. This score suggests that the synthetic data maintains a good representation of the relationships between columns, although not perfect.


**Overall Score: 90.5%**
The overall score combines the scores from both Data Validity and Data Structure evaluations. In this case, it is calculated as an average or a weighted average of the two scores, resulting in an overall score of 90.5%.

Nevertheless, the "Attrition" feature remains significantly imbalanced. Despite addressing privacy concerns, we aim to rectify this imbalance for improved prediction accuracy. To achieve this, we will impose a condition to create a dataset with an equal distribution of "Attrition," comprising 50% "Yes" and 50% "No."

## Generation of Synthetic Data Using SDV - Balancing the Target Variable

In [ ]:
attrition = Condition(
    num_rows=2000,
    column_values={'Attrition': 'Yes'}
)

no_attrition = Condition(
    num_rows=2000,
    column_values={'Attrition': 'No'}
)

In [ ]:
synthetic_data2 = synthesizer.sample_from_conditions(
    conditions=[attrition, no_attrition],
    output_file_path='synthetic_simulated_scenario.csv'
)

In [ ]:
synthetic_data2.head()

In [ ]:
diagnostic = run_diagnostic(
    real_data=df,
    synthetic_data=synthetic_data2,
    metadata=metadata
)

In [ ]:
quality_report = evaluate_quality(
    df,
    synthetic_data2,
    metadata
)

In [ ]:
quality_report.get_details('Column Shapes')

In [ ]:
for col in df.columns:
    fig = get_column_plot(
        real_data=df,
        synthetic_data=synthetic_data2,
        column_name=col,
        metadata=metadata
    )

    fig.show()

In [ ]:
comparison_column = 'Attrition'

for col in df.columns:
    if col != comparison_column:
        fig = get_column_pair_plot(
            real_data=df,
            synthetic_data=synthetic_data2,
            column_names=[comparison_column, col],
            metadata=metadata
        )

        fig.show()

**First Report:**
* Data Validity: 100.0%

This indicates that the synthetic data generated by SDV adheres to the specified constraints and validity checks. All records in the synthetic dataset meet the required criteria, ensuring the validity of the data.

* Data Structure: 100.0%

The score of 100.0% suggests that the synthetic data accurately preserves the overall structure of the original data. This includes maintaining relationships, dependencies, and patterns present in the original dataset.


**Second Report:**
* Column Shapes: 90.53%

This score evaluates how well the synthetic data reproduces the shapes or distributions of individual columns. A high score, in this case, 90.53%, implies that the statistical properties of each column in the synthetic data closely resemble those of the corresponding columns in the original dataset.

* Column Pair Trends: 85.26%

This score assesses how well the relationships between pairs of columns are captured in the synthetic data. It looks at trends and correlations between pairs of columns. An 85.26% score indicates that the synthetic data maintains a reasonably good representation of the relationships between columns, though not perfect.


**Overall Score: 87.9%

The overall score is likely calculated as an average or a weighted average of the scores from Data Validity and Data Structure evaluations. In this case, it results in an overall score of 87.9%.

The overall score observed in this synthetic dataset is less than the score achieved in the initial synthetic data generation. Additionally, although "Attrition" is evenly distributed, the distribution of the remaining variables closely mirrors that of the first dataset, raising concerns. This suggests potential instances where "Yes" might have been artificially imposed, resulting in a lack of true representation for "Attrition" == "Yes" in our original data. This suspicion is reinforced by the high score in Column Shapes and a lower score in Column Pair Trends, indicating potential misrepresentation of observations where "Attrition" is labeled as "Yes."

## Synthetic Data 1 - Modelling

In [ ]:
synthetic_data.head()

In [ ]:
synthetic_data.describe()

In [ ]:
datasets = skf(synthetic_data.copy(), catvars, target)

study = optuna.create_study(direction='maximize')
study.optimize(lambda trial: lgb_objective(trial, datasets.copy()), n_trials=5, n_jobs=2)
best_params = study.best_params
best_trial = study.best_trial

In [ ]:
print(f"Best trial - Loss: {best_trial.value:.4f}, \n Params: {best_params}")

In [ ]:
fig = optuna.visualization.plot_param_importances(study)
fig.show()

In [ ]:
best_params

In [ ]:
train, test = train_test(synthetic_data.copy(), catvars, target)
model = LGBMClassifier(**best_params, verbose=-1)
early_stopping_callback = lgb.early_stopping(stopping_rounds=20, verbose=True)
model.fit(train.data, train.label, eval_metric='auc')

In [ ]:
evaluate(model, train.data, test_data_compare, train.label, test_label_compare)

In [ ]:
plot_importance(model, importance_type='split', figsize=(10, 8))
plt.show()

## Synthetic Data 2 - Modelling

In [ ]:
synthetic_data2.head()

In [ ]:
synthetic_data2.describe()

In [ ]:
datasets = skf(synthetic_data2.copy(), catvars, target)

study = optuna.create_study(direction='maximize')
study.optimize(lambda trial: lgb_objective(trial, datasets.copy()), n_trials=5, n_jobs=2)
best_params = study.best_params
best_trial = study.best_trial

In [ ]:
print(f"Best trial - Loss: {best_trial.value:.4f}, \n Params: {best_params}")

In [ ]:
fig = optuna.visualization.plot_param_importances(study)
fig.show()

In [ ]:
best_params

In [ ]:
train, test = train_test(synthetic_data2.copy(), catvars, target)
model = LGBMClassifier(**best_params, verbose=-1)
early_stopping_callback = lgb.early_stopping(stopping_rounds=20, verbose=True)
model.fit(train.data, train.label, eval_metric='auc')

In [ ]:
evaluate(model, train.data, test_data_compare, train.label, test_label_compare)

In [ ]:
plot_importance(model, importance_type='split', figsize=(10, 8))
plt.show()

The comparison was conducted between the first and second synthetic datasets. Despite the default label being "Real," it's essential to note that the data used for this comparison was synthetic. The objective was to scrutinize the dissimilarities in distribution between the two datasets and confirm the accurate generation of observations where "Attrition" is labeled as "Yes." The analysis suggests that individuals with lower wages, younger age, and less tenure in the company are more likely to leave. As a result, we can infer that the synthetic_data2 effectively captures an authentic representation of both "Attrition" values.

In [ ]:
for col in df.columns:
    fig = get_column_plot(
        real_data=synthetic_data,
        synthetic_data=synthetic_data2,
        column_name= col,
        metadata=metadata
    )

    fig.show()

# Hybrid Dataset - Real and Synthetic Data
Given the superior precision and recall scores in the latest synthetic dataset, we have chosen to use it. This dataset will be employed in conjunction with the original dataset, creating a hybrid dataset comprising both real and synthetic data.

In [ ]:
df.head()

In [ ]:
synthetic_data2.head()

In [ ]:
# Concatenate the DataFrames
mix_df = pd.concat([df, synthetic_data2], ignore_index=True)

In [ ]:
mix_df.head()

In [ ]:
mix_df.describe()

## Hybrid Dataset - Modelling

In [ ]:
datasets = skf(mix_df.copy(), catvars, target)

study = optuna.create_study(direction='maximize')
study.optimize(lambda trial: lgb_objective(trial, datasets.copy()), n_trials=5, n_jobs=2)
best_params = study.best_params
best_trial = study.best_trial

In [ ]:
print(f"Best trial - Loss: {best_trial.value:.4f}, \n Params: {best_params}")

In [ ]:
fig = optuna.visualization.plot_param_importances(study)
fig.show()

In [ ]:
best_params

In [ ]:
train, test = train_test(mix_df.copy(), catvars, target)
model = LGBMClassifier(**best_params, verbose=-1)
early_stopping_callback = lgb.early_stopping(stopping_rounds=20, verbose=True)
model.fit(train.data, train.label, eval_metric='auc')

In [ ]:
evaluate(model, train.data, test_data_compare, train.label, test_label_compare)

In [ ]:
plot_importance(model, importance_type='split', figsize=(10, 8))
plt.show()

# Anonymized Data

In [ ]:
df_anon = pd.read_csv('/content/anonK6L4.csv')

In [ ]:
df_anon.head()

In [ ]:
df_anon.describe()

In [ ]:
df_anon.info()

## Drop based on Unique

In [ ]:
col_unique = df_anon.nunique().sort_values()
col_unique

In [ ]:
# Filter columns with only 1 unique value
cols_to_drop = col_unique[col_unique == 1].index

# Drop columns from df_anon
df_anon.drop(cols_to_drop, axis=1, inplace=True)

In [ ]:
df_anon.info()

## Explore Based on Types

In [ ]:
cat_treshold = 20

In [ ]:
catvars_complex = []
catvars = []
numvars = []
ordvars = []
target= 'Attrition'
for column in df_anon.columns:
    if column == target:
        continue
    if df_anon[column].dtype == object:
        if col_unique[column] > cat_treshold:
            catvars_complex.append(column)
        else:
            catvars.append(column)
    else:
        if col_unique[column] > cat_treshold:
            numvars.append(column)
        else:
            ordvars.append(column)

In [ ]:
print('Catvars Complex:', catvars_complex)
print('\nCatvars:', catvars)
print('\nNumerical:', numvars)
print('\nOrdvars:', ordvars)

## Numericals

In [ ]:
# Analyzing Numerical Variables
sns.set(style="whitegrid")

fig, axes = plt.subplots(nrows=len(numvars), ncols=1, figsize=(10, len(numvars)*5))
for i, var in enumerate(numvars):
    sns.histplot(df_anon[var], ax=axes[i])
    axes[i].set_title(f'Distribution of {var}')
plt.tight_layout()
plt.show()

In [ ]:
cat_ord_vars = catvars + ordvars
cat_ord_size = len(cat_ord_vars)
cat_ord_dict = {}
for key in cat_ord_vars:
    cat_ord_dict[key] = df_anon[key].values

In [ ]:
fig, axes = plt.subplots(nrows=cat_ord_size-1, ncols=1, figsize=(10, cat_ord_size*4))

# Exclude the variable "Type" from the list of cat_ord_vars
cat_ord_vars_except_type = [var for var in cat_ord_vars if var != 'Type']

for i, var in enumerate(cat_ord_vars_except_type):
    sns.countplot(x=var, data=cat_ord_dict, ax=axes[i])
    axes[i].set_title(f'Distribution of {var}', fontsize=14, fontweight='bold')  # Adjust title size and boldness
    axes[i].tick_params(axis='x', rotation=45, labelsize=10)  # Rotate x-axis labels to avoid label overlap

plt.tight_layout()
plt.show()

In [ ]:
cat_vars_to_be_binned = ['Age', 'BusinessTravel', 'Gender', 'OverTime', 'PerformanceRating', 'RelationshipSatisfaction', 'WorkLifeBalance']

In [ ]:
le = LabelEncoder()
for col in cat_vars_to_be_binned:
    df_anon[col] = le.fit_transform(df_anon[col])

In [ ]:
df_anon.head()

In [ ]:
df_anon.info()

## Target

In [ ]:
# Create a bar plot for the 'Attrition' feature
plt.figure(figsize=(8, 6))
ax = sns.countplot(x='Attrition', data=df_anon, palette='viridis')

# Add total counts and percentages on top of the bars
total_counts = len(df)
for p in ax.patches:
    height = p.get_height()
    ax.annotate(f'{int(height)}\n{height/total_counts*100:.2f}%',
                (p.get_x() + p.get_width() / 2., height),
                ha='center', va='center', xytext=(0, 10),
                textcoords='offset points', fontsize=12, color='black')

plt.title('Attrition Count')
plt.xlabel('Attrition')
plt.ylabel('Count')
plt.show()

### Against Target

In [ ]:
# same to_dict problem
df_cols = df_anon.columns
df_size = len(df_cols)
df_dict = {}
for key in df_cols:
    df_dict[key] = df_anon[key].values

In [ ]:
cat_ord_target_dict = {**cat_ord_dict, target: df_anon[target].values}

In [ ]:
# Exclude the variable "Type" from the list of cat_ord_vars
cat_ord_target_dict_except_type = [var for var in cat_ord_target_dict if var != 'Type']

fig, axes = plt.subplots(nrows=len(cat_ord_target_dict_except_type), ncols=1, figsize=(10, cat_ord_size*4))

for i, var in enumerate(cat_ord_target_dict_except_type):
    sns.histplot(
        data=cat_ord_target_dict, x=var,  hue=target,
        binwidth=0.5, stat="count", multiple="stack",
        ax=axes[i]
    )
plt.tight_layout()
plt.show()

## Modelling Anonymized Data

In [ ]:
# Assuming 'type' is the column distinguishing between 'Train' and 'Test'
df_anon_train = df_anon[df_anon['Type'] == 'Train'].copy()
df_anon_test = df_anon[df_anon['Type'] == 'Test'].copy()

# Drop the 'type' column from both df_anon_train and df_anon_test
df_anon_train.drop(columns=['Type'], inplace=True)
df_anon_test.drop(columns=['Type'], inplace=True)


In [ ]:
datasets = skf(df_anon_train.copy(), catvars, target)

study = optuna.create_study(direction='maximize')
study.optimize(lambda trial: lgb_objective(trial, datasets.copy()), n_trials=5, n_jobs=2)
best_params = study.best_params
best_trial = study.best_trial

In [ ]:
print(f"Best trial - Loss: {best_trial.value:.4f}, \n Params: {best_params}")

In [ ]:
fig = optuna.visualization.plot_param_importances(study)
fig.show()

In [ ]:
best_params

In [ ]:
train, test = train_test(df_anon_train.copy(), catvars, target)
model = LGBMClassifier(**best_params, verbose=-1)
early_stopping_callback = lgb.early_stopping(stopping_rounds=20, verbose=True)
model.fit(train.data, train.label, eval_metric='auc')

In [ ]:
test_label = df_anon_test['Attrition'].copy()
test_data = df_anon_test.drop(columns=['Attrition']).copy()

In [ ]:
evaluate(model, train.data, test_data, train.label, test_label)

In [ ]:
plot_importance(model, importance_type='split', figsize=(10, 8))
plt.show()

Because of the imbalance in the target variable, the model performs well when predicting "Attrition" == "No" but struggles with positive outcomes, which are more critical to the problem at hand. To address this issue, we will implement SMOTE (Synthetic Minority Over-sampling Technique).

## Modelling Anonymized Data with the Use of SMOTE

In [ ]:
# Instantiate SMOTE
smote = SMOTE(sampling_strategy=0.5, random_state=42)

In [ ]:
# Apply SMOTE to the training set
X_train_resampled_anon, y_train_resampled_anon = smote.fit_resample(train.data, train.label)

In [ ]:
df_anon_smote = pd.concat([X_train_resampled_anon, y_train_resampled_anon], axis=1)

In [ ]:
sns.countplot(x='Attrition', data=df_anon_smote)
plt.title('Distribution of Attrition', fontsize=14, fontweight='bold')
plt.show()

In [ ]:
datasets = skf(df_anon_smote.copy(), catvars, target)

study = optuna.create_study(direction='maximize')
study.optimize(lambda trial: lgb_objective(trial, datasets.copy()), n_trials=5, n_jobs=2)
best_params = study.best_params
best_trial = study.best_trial

In [ ]:
print(f"Best trial - Loss: {best_trial.value:.4f}, \n Params: {best_params}")

In [ ]:
fig = optuna.visualization.plot_param_importances(study)
fig.show()

In [ ]:
best_params

In [ ]:
train, test = train_test(df_anon_smote.copy(), catvars, target)
model = LGBMClassifier(**best_params, verbose=-1)
early_stopping_callback = lgb.early_stopping(stopping_rounds=20, verbose=True)
model.fit(train.data, train.label, eval_metric='auc')

In [ ]:
evaluate(model, train.data, test_data, train.label, test_label)

In [ ]:
plot_importance(model, importance_type='split', figsize=(10, 8))
plt.show()

Regrettably, the outcomes did not show improvement. SMOTE did not prove to be beneficial in this particular case. In high-dimensional feature spaces, SMOTE may struggle to create meaningful synthetic samples, as the curse of dimensionality can lead to sparsity and difficulty in identifying relevant nearest neighbors. Furthermore, some classifiers, especially those that are inherently good at handling imbalanced datasets, might not benefit significantly from SMOTE.